In [ ]:
# graph_utils.py
from __future__ import annotations

from dataclasses import dataclass, field
from typing import Dict, List, Optional, Tuple, Iterable, Any
import numpy as np


# ----------------------------
# Constants / Tags / Edge Types
# ----------------------------

TAG_DOOR = "door"  # 建议 door 独立成一个 domain/tag

EDGE_INTRA = "intra"    # 同域边：indoor-indoor / outdoor-outdoor / door-door
EDGE_PORTAL = "portal"  # door 跨域边：door-indoor / door-outdoor


# ----------------------------
# Node / Edge dataclasses
# ----------------------------

@dataclass(frozen=True)
class Node:
    """Topology node in continuous 2D space."""
    id: int
    x: float
    y: float

    tag: str                 # "indoor" / "outdoor" / "door" / "no-traversable" / ...
    score: float             # traversability score at this point (or default)

    region_id: Optional[str] = None  # 可选：属于哪个具体 Region 实例（如果你需要更细粒度）
    source: str = "random"           # "random" / "forced_door" / ...


@dataclass(frozen=True)
class Edge:
    """Graph edge with semantic type."""
    id: int
    u: int
    v: int
    w: float                 # edge weight (e.g., Euclidean distance)

    etype: str = EDGE_INTRA  # "intra" or "portal"
    reason: Optional[str] = None     # 调试：为什么存在/或为什么被拒绝（若你后续记录拒绝边）
    meta: Dict[str, Any] = field(default_factory=dict)  # 扩展字段：length、cost components等


# ----------------------------
# Graph container
# ----------------------------

class Graph:
    """
    Graph stores:
      - nodes: list[Node] indexed by node.id
      - edges: list[Edge] indexed by edge.id
      - adj: adjacency list where adj[u] = list[(v, w, edge_id)]
      - indices: node_ids_by_tag, edge_ids_by_type
    """

    def __init__(self) -> None:
        self.nodes: List[Node] = []
        self.edges: List[Edge] = []
        self.adj: List[List[Tuple[int, float, int]]] = []

        # indices/caches
        self.node_ids_by_tag: Dict[str, List[int]] = {}
        self.edge_ids_by_type: Dict[str, List[int]] = {}

    # ---- node ops ----
    def add_node(
        self,
        x: float,
        y: float,
        tag: str,
        score: float,
        region_id: Optional[str] = None,
        source: str = "random",
    ) -> int:
        node_id = len(self.nodes)
        n = Node(id=node_id, x=float(x), y=float(y), tag=str(tag), score=float(score),
                 region_id=region_id, source=str(source))
        self.nodes.append(n)
        self.adj.append([])

        self.node_ids_by_tag.setdefault(n.tag, []).append(node_id)
        return node_id

    # ---- edge ops ----
    def add_edge(
        self,
        u: int,
        v: int,
        w: float,
        etype: str = EDGE_INTRA,
        reason: Optional[str] = None,
        meta: Optional[Dict[str, Any]] = None,
        undirected: bool = True,
    ) -> int:
        """
        Add an edge. By default graph is undirected: adds both (u->v) and (v->u) in adj list,
        but only one Edge object is stored.
        """
        if u == v:
            raise ValueError("Self-edge is not allowed.")
        if not (0 <= u < len(self.nodes)) or not (0 <= v < len(self.nodes)):
            raise IndexError("Node index out of range.")

        edge_id = len(self.edges)
        e = Edge(
            id=edge_id,
            u=int(u),
            v=int(v),
            w=float(w),
            etype=str(etype),
            reason=reason,
            meta={} if meta is None else dict(meta),
        )
        self.edges.append(e)

        self.adj[u].append((v, float(w), edge_id))
        if undirected:
            self.adj[v].append((u, float(w), edge_id))

        self.edge_ids_by_type.setdefault(e.etype, []).append(edge_id)
        return edge_id

    # ---- query helpers ----
    def neighbors(self, u: int) -> List[Tuple[int, float, int]]:
        return self.adj[u]

    def get_node(self, node_id: int) -> Node:
        return self.nodes[node_id]

    def get_edge(self, edge_id: int) -> Edge:
        return self.edges[edge_id]

    def node_ids(self, tag: Optional[str] = None) -> List[int]:
        if tag is None:
            return list(range(len(self.nodes)))
        return self.node_ids_by_tag.get(tag, [])

    def edge_ids(self, etype: Optional[str] = None) -> List[int]:
        if etype is None:
            return list(range(len(self.edges)))
        return self.edge_ids_by_type.get(etype, [])

    def __len__(self) -> int:
        return len(self.nodes)


# ----------------------------
# NodeSampler
# ----------------------------

class NodeSampler:
    """
    Sample nodes in continuous 2D space under environment constraints.

    Required interface from RegionManager (from map_utils.py):
      - query_tag(x, y, default="unknown") -> str
      - query_score(x, y, default=1.0) -> float

    You can force sample door nodes from a separate door RegionManager (dm),
    and label them with tag 'door' (or whatever dm returns).
    """

    def __init__(
        self,
        width: float,
        height: float,
        region_manager,             # main RegionManager (indoor/outdoor/obstacles)
        door_manager=None,          # optional: door RegionManager
        *,
        default_score: float = 1.0,
        default_tag: str = "unknown",
        no_traversable_tag: str = "no-traversable",
        rng_seed: int = 0,
    ) -> None:
        self.W = float(width)
        self.H = float(height)

        self.rm = region_manager
        self.dm = door_manager

        self.default_score = float(default_score)
        self.default_tag = str(default_tag)
        self.no_trav_tag = str(no_traversable_tag)

        self.rng = np.random.default_rng(rng_seed)

    # ---- basic environment queries ----
    def _in_bounds(self, x: float, y: float) -> bool:
        return (0.0 <= x < self.W) and (0.0 <= y < self.H)

    def _query_tag_score(self, x: float, y: float) -> Tuple[str, float]:
        tag = self.rm.query_tag(x, y, default=self.default_tag)
        score = self.rm.query_score(x, y, default=self.default_score)
        return tag, score

    def _is_free(self, tag: str, score: float) -> bool:
        # 你 map_utils 里 obstacle 可能用 tag 或 score=inf 表达，两者都兜住
        if tag == self.no_trav_tag:
            return False
        if not np.isfinite(score):
            return False
        return True

    # ---- sampling primitives ----
    def sample_random_nodes(
        self,
        n: int,
        *,
        allowed_tags: Optional[Iterable[str]] = None,
        max_tries: int = 100000,
        source: str = "random",
    ) -> List[Tuple[float, float, str, float]]:
        """
        Uniformly sample n nodes across the whole map (continuous),
        reject if out of bounds or not free or not in allowed_tags.
        Returns: list of (x, y, tag, score)
        """
        allowed_set = set(allowed_tags) if allowed_tags is not None else None

        out: List[Tuple[float, float, str, float]] = []
        tries = 0
        while len(out) < n and tries < max_tries:
            tries += 1
            x = float(self.rng.uniform(0.0, self.W))
            y = float(self.rng.uniform(0.0, self.H))

            if not self._in_bounds(x, y):
                continue

            tag, score = self._query_tag_score(x, y)
            if not self._is_free(tag, score):
                continue
            if allowed_set is not None and tag not in allowed_set:
                continue

            out.append((x, y, tag, score))

        if len(out) < n:
            raise RuntimeError(
                f"Failed to sample {n} nodes (got {len(out)}) after {tries} tries. "
                f"Consider relaxing constraints or increasing max_tries."
            )
        return out

    def sample_door_nodes(
        self,
        n: int,
        *,
        max_tries: int = 200000,
        door_tag: str = TAG_DOOR,
        source: str = "forced_door",
    ) -> List[Tuple[float, float, str, float]]:
        """
        Force sample n nodes inside door region(s).
        Logic:
          - Use door_manager.dm to test membership by tag query.
          - For sampled point in door, still require it's free under main rm (not obstacle).
        Returns: list of (x, y, tag, score) with tag = door_tag by default.
        """
        if self.dm is None:
            raise ValueError("door_manager (dm) is None but sample_door_nodes() was called.")

        out: List[Tuple[float, float, str, float]] = []
        tries = 0
        while len(out) < n and tries < max_tries:
            tries += 1
            x = float(self.rng.uniform(0.0, self.W))
            y = float(self.rng.uniform(0.0, self.H))
            if not self._in_bounds(x, y):
                continue

            # 判断是否在 door 范围：由 door manager 决定
            dtag = self.dm.query_tag(x, y, default="unknown")
            if dtag != door_tag:
                # 如果你 door manager 返回的不是 "door"，那这里改成 “dtag in {...}”
                continue

            # door 点也要在主地图中可通行（不在 obstacle）
            tag_main, score_main = self._query_tag_score(x, y)
            if not self._is_free(tag_main, score_main):
                continue

            # door 节点最终 tag 设为 door_tag，score 用主地图 score（更合理）
            out.append((x, y, door_tag, score_main))

        if len(out) < n:
            raise RuntimeError(
                f"Failed to sample {n} door nodes (got {len(out)}) after {tries} tries. "
                f"Check your door regions/tags or increase door area / max_tries."
            )
        return out

    # ---- convenience: create a Graph with sampled nodes ----
    def make_graph_with_nodes(
        self,
        n_random: int,
        n_door: int = 0,
        *,
        random_allowed_tags: Optional[Iterable[str]] = None,
    ) -> Graph:
        """
        Create a Graph containing sampled nodes only (no edges yet).
        """
        g = Graph()

        # random nodes (typically indoor/outdoor)
        random_samples = self.sample_random_nodes(
            n_random,
            allowed_tags=random_allowed_tags,
            source="random",
        )
        for x, y, tag, score in random_samples:
            g.add_node(x=x, y=y, tag=tag, score=score, source="random")

        # forced door nodes
        if n_door > 0:
            door_samples = self.sample_door_nodes(n_door, source="forced_door")
            for x, y, tag, score in door_samples:
                g.add_node(x=x, y=y, tag=tag, score=score, source="forced_door")

        return g
